## Station Status Exploration Notebook

**Objective:** Determine Which of the information now available can be used to predict either bike stockouts or
full stations at some future future time interval. This notebook uses development data: the gold tables,
restricted to a finite time range defined at the top of this notebook (`RESEARCH_START` and `RESEARCH_END`).

We focus on four separate areas:

1. Temporal stockout patterns
2. Geographic patterns
3. Correlations/Associations of Variables with Stockouts
4. Fit some basic Logistic Regression Models and examine them


In [0]:
import textwrap

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize, TwoSlopeNorm
from matplotlib.cm import ScalarMappable
from pyspark.sql import functions as F

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler

spark.conf.set("spark.sql.session.timeZone", "UTC")

# The research period: a fixed time range of the gold tables
RESEARCH_START = "2026-08-21T04:00:00Z"   # New York midnight, August 21
RESEARCH_END = "2026-09-28T04:00:00Z"     # New York midnight, September 28 (exclusive)
VALIDATION_START = "2026-09-21T04:00:00Z"  # We will have a separate validation set

HORIZON_MIN = 30 # This is not the world's most ambitious choice

# Model settings, matching the training notebook
SAMPLE_FRACTION = 0.1             # Share of training rows brought into pandas
VALIDATION_SAMPLE_FRACTION = 0.5  # Share of validation rows brought into pandas
C_PENALTY = 0.25                  # C = tau^2, with the prior scale tau = 0.5 chosen by cross-validation
MAX_ITER = 500

# Names of the gold tables
PREDICTOR_TABLE = "workspace.default.gold_station_predictors"
TARGET_TABLE = "workspace.default.gold_station_targets"

research_start = F.to_timestamp(F.lit(RESEARCH_START))
research_end = F.to_timestamp(F.lit(RESEARCH_END))

## 1. Data Preparation

The gold tables already contain what this notebook needs. `build_predictors` and `build_targets` in
`station_features.py` make them, for the gold notebook:

1. Checks that there is no duplicate data
2. Joins with station_info to add contextual/temporally appropriate information about the stations
3. Creates the stockout variables, 'bike_stockout' or 'dock_stockout', the predictors, and the targets

The main data quality check in gold is to set the target to be is null unless the gaps between polls in the interval from prediction time to horizon time of less than 10 minutes, and there is at least one poll post horizon within the first 10 mintues.

In [0]:
# The weekly job rebuilds gold. We track the version of the data that we use to guarantee replicability

gold_versions = {
    table: spark.sql(f"DESCRIBE HISTORY {table} LIMIT 1").first()["version"]
    for table in [PREDICTOR_TABLE, TARGET_TABLE]
}
print("Gold versions:", gold_versions)

# Load data and targets

gold_predictors = spark.read.option("versionAsOf", gold_versions[PREDICTOR_TABLE]).table(PREDICTOR_TABLE)
gold_targets = (spark.read.option("versionAsOf", gold_versions[TARGET_TABLE]).table(TARGET_TABLE)
                .filter(F.col("horizon_minutes") == HORIZON_MIN))

# We are going to use these to calculate overall station capacity 

component_columns = [
    "num_bikes_available", "num_bikes_disabled",
    "num_docks_available", "num_docks_disabled",
]

# observations is the data frame that we are going to use for analysis.
# We add some basic features to it, like the "reported total"
# There will be some times in the data where the reported total is inconsistent
# with the official station capacity. The causes of this could be manifold, 
# I have identified station valet service and other moving operations as causes
# Regretably, despite GBFS 2.3 having a "is_valet" column citibike seems not to
# transmit it

observations = (
    gold_predictors
    .filter((F.col("fetched_at") >= research_start) & (F.col("fetched_at") < research_end))
    .withColumn(
        "reported_total",
        sum(F.col(c).cast("double") for c in component_columns),
    )
    .withColumn("capacity_discrepancy", F.col("reported_total") - F.col("capacity")) # This measures capacity inconsistencies
)

# Each row has its predictors and its targets
research_features = observations.join(
    gold_targets.drop("horizon_minutes"),
    ["station_id", "fetched_at"],
    "left",
)


# Helper function for graphing:

# We are making lots of maps so we try to be efficient with code


background_stations = (
    observations
    .filter("lat IS NOT NULL AND lon IS NOT NULL")
    .select("lat", "lon").distinct()
    .toPandas()
)

def map_background(ax):
    ax.set_facecolor("#f4f4f4")
    ax.scatter(background_stations.lon, background_stations.lat,
               s=3, color="#d0d0d0", zorder=1, linewidths=0)
    ax.set_aspect(1 / np.cos(np.radians(40.73)))
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_xticks([])
    ax.set_yticks([])


# This gives a quick summary of our data

display(observations.agg(
    F.min("fetched_at").alias("first_poll"),
    F.max("fetched_at").alias("last_poll"),
    F.count("*").alias("polls"),
    F.countDistinct("station_id").alias("stations"),
    F.sum(F.col("info_at").isNull().cast("long")).alias("missing_historical_info"),
    F.count("bike_stockout").alias("eligible_bike_polls"),
    F.count("dock_stockout").alias("eligible_dock_polls"),
    F.avg(F.col("bike_stockout").cast("double")).alias("bike_stockout_share"),
    F.avg(F.col("dock_stockout").cast("double")).alias("dock_stockout_share"),
))

## Capacity Inconsistencies

As mentioned in the previous slide, the total number of bikes and docks at a station can sometimes exceed that stations official capacity. I have 
strong suspicions about what some of these events are caused by, for example valet service where a person arrives at a very busy station with a huge truck either full or bikes or which can be filled with bikes, to temporarily enhance station capacity. Citibike notably mentions a few stations which regularly employ this service. Unfortunately, I have yet to find a flag which proves it is what happens. This is worth further investigation

In [0]:

# How often does the reported total exceed historical capacity?
capacity_check = observations.filter(
    "capacity > 0 AND reported_total IS NOT NULL"
)

display(capacity_check.agg(
    F.count("*").alias("comparable_polls"),
    (100 * F.avg(
        (F.col("capacity_discrepancy") > 0).cast("double")
    )).alias("percent_over_capacity"),
    F.max("capacity_discrepancy").alias("largest_excess"),
))

# Inspect the largest discrepancies and their component counts.
display(
    capacity_check
    .filter("capacity_discrepancy > 0")
    .select(
    "station_id", "station_name", "fetched_at", "capacity",
    *component_columns, "reported_total", "capacity_discrepancy",
)
    .orderBy(F.desc("capacity_discrepancy"))
    .limit(10)
)

## 2. Temporal and geographic patterns
Two figures describe current shortages, not future onset. Rates use separate eligibility denominators.
Geographic differences compare the same historical station locations; they are not matched day-by-day.

## Temporal Patterns:

1. No bikes is rarer than no docks by a factor of 3
2. No bikes peaks just after the morning commute on weekdays and only gets resolved overnight
3. No docks builds in the afternoon and peaks in the evening. Again it is resolved early in the morning, with a small secondary peak just past the morning commute
4. Weekend patterns are shifted later in the day

In [0]:
hourly = (
    observations.groupBy("hour", "weekend")
    .agg(*[
        F.avg(F.col(f"{k}_stockout").cast("double")).alias(k)
        for k in ["bike", "dock"]
    ])
    .toPandas()
)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), constrained_layout=True)
for ax, outcome in zip(axes, ["bike", "dock"]):
    for weekend, name in [(0, "Weekday"), (1, "Weekend")]:
        q = hourly[hourly.weekend == weekend].sort_values("hour")
        ax.plot(q.hour, 100*q[outcome], label=name)
    ax.set(
        title=f"No {outcome}s",
        xlabel="New York hour",
        ylabel="Eligible polls (%)",
    )
    ax.legend()
plt.show()

In [0]:
geography = (
    observations
    .filter("lat IS NOT NULL AND lon IS NOT NULL")
    .withColumn("period", F.floor(F.col("hour") / 6))
    .groupBy("station_id", "lat", "lon", "period")
    .agg(
        F.count("bike_stockout").alias("bike_polls"),
        F.avg(F.col("bike_stockout").cast("double")).alias("bike_rate"),
        F.count("dock_stockout").alias("dock_polls"),
        F.avg(F.col("dock_stockout").cast("double")).alias("dock_rate"),
    )
    .toPandas()
)

period_names = [
    "Midnight–6am",
    "6am–noon",
    "Noon–6pm",
    "6pm–midnight",
]

fig, axes = plt.subplots(
    2, 4,
    figsize=(22, 15),
    constrained_layout=True,
)

rate_scale = Normalize(0, 50)

for row, outcome in enumerate(["bike", "dock"]):
    for period, period_name in enumerate(period_names):
        ax = axes[row, period]
        map_background(ax)

        q = geography[
            (geography.period == period)
        ].sort_values(f"{outcome}_rate")

        ax.scatter(
            q.lon, q.lat,
            c=100 * q[f"{outcome}_rate"],
            cmap="magma_r", norm=rate_scale,
            s=8, zorder=2,
        )
        ax.set_title(
            f"No {outcome}s: {period_name}",    
            fontsize=18,
            pad=10
        )


colorbar = fig.colorbar(
    ScalarMappable(norm=rate_scale, cmap="magma_r"),
    ax=axes.ravel().tolist(),
    shrink=.6,
    extend="max",
)
colorbar.set_label("Eligible polls with a shortage (%)", fontsize=16)
colorbar.ax.tick_params(labelsize=14)

fig.suptitle(
    "Geographic shortage prevalence by local time",
    fontsize=26,
    fontweight="bold",
    x=0.05, y=0.99,
    ha="left",
)

fig.text(
    0.05, 0.95,
    "There are distinct patterns of outages. There is a persistent shortage of bikes in\n"
    "Midtown, Harlem/Upper Manhattan, the Bronx, and Brooklyn around Prospect Park. This shortage is only\n"
    "alleviated in Midtown after the morning commute. There is an extreme shortage of dock space\n"
    "near hospitals on the Upper East Side, in Williamsburg and Greenpoint, and along the\n"
    "peripheral boundary of the network in Brooklyn and Queens.",
    fontsize=14,
    ha="left",
    va="top",
    linespacing=1.4,
)

# Reserve space above the panels for the title and subtitle.
fig.get_layout_engine().set(rect=(0, 0, 1, 0.80))

plt.show()

In [0]:
fullness_geography = (
    observations
    .filter(
        "lat IS NOT NULL AND lon IS NOT NULL "
        "AND capacity > 0 AND bike_stockout IS NOT NULL"
    )
    .withColumn("period", F.floor(F.col("hour") / 6))
    .withColumn(
        "fullness",
        F.least(
            F.col("num_bikes_available") / F.col("capacity"),
            F.lit(1.0),
        ),
    )
    .groupBy("station_id", "lat", "lon", "period")
    .agg(F.avg("fullness").alias("mean_fullness"))
    .toPandas()
)

fig, axes = plt.subplots(
    1, 4,
    figsize=(22, 9),
    constrained_layout=True,
)

fullness_scale = Normalize(0, 1)

for ax, period, period_name in zip(axes, range(4), period_names):
    map_background(ax)

    q = fullness_geography[
        fullness_geography.period == period
    ].sort_values("mean_fullness")

    ax.scatter(
        q.lon, q.lat,
        c=q.mean_fullness,
        cmap="viridis", norm=fullness_scale,
        s=8, zorder=2,
    )
    ax.set_title(
        period_name,
        fontsize=18,
        pad=10,
    )

colorbar = fig.colorbar(
    ScalarMappable(norm=fullness_scale, cmap="viridis"),
    ax=axes.tolist(),
    shrink=.6,
)
colorbar.set_label("Mean available bikes / capacity", fontsize=16)
colorbar.ax.tick_params(labelsize=14)

fig.suptitle(
    "Mean station fullness by local time",
    fontsize=26,
    fontweight="bold",
    x=0.05, y=0.99,
    ha="left",
)

fig.text(
    0.05, 0.93,
    textwrap.fill(
        "Overall capacity, just like stockouts, exhibits substantial geographic heterogeneity. "
        "Geograpraphic influence persists in all time periods, some areas are permanently short bikes and others docks.",
        width=140,
    ),
    fontsize=16,
    ha="left",
    va="top",
    linespacing=1.4,
)

fig.get_layout_engine().set(rect=(0, 0, 1, 0.80))

plt.show()

## Model Exploration

We will now build some basic models of stockouts, using our gold data.

### Predictors

This is a description of the features that we are using

| Predictor | Meaning |
|---|---|
| `bike_available_fraction` | Available bikes / capacity |
| `dock_available_fraction` | Available docks / capacity |
| `bike_change_per_minute` | Bike change since the previous poll, per minute |
| `dock_change_per_minute` | Dock change since the previous poll, per minute |
| `bike_change_30min_per_minute` | Net bike change over approximately 30 minutes, per minute |
| `dock_change_30min_per_minute` | Net dock change over approximately 30 minutes, per minute |
| `bike_change_per_minute_fraction` | Latest bike change per minute ÷ capacity |
| `dock_change_per_minute_fraction` | Latest dock change per minute ÷ capacity |
| `bike_change_30min_per_minute_fraction` | Longer-window bike change per minute ÷ capacity |
| `dock_change_30min_per_minute_fraction` | Longer-window dock change per minute ÷ capacity |
| `hour_sin_1`, `hour_cos_1` | 24-hour cycle |
| `hour_sin_2`, `hour_cos_2` | 12-hour cycle |
| `log_capacity` | Log of  station capacity |
| `geographic_cell` | Approximate kilometre-scale location category. x and y grid coordinates, very crude |

These predictors are computed by `build_predictors` in `station_features.py`, when the gold notebook builds the gold tables.

# Validation Set

We will construct a one week validation set. We don't have that much more data at the moment.

In [0]:
split_microseconds = F.expr(
    f"unix_micros(to_timestamp('{VALIDATION_START}'))"
)

# Training labels must be complete before the validation week begins
training_features = research_features.filter(
    F.col("label_end_microseconds") < split_microseconds
)

validation_features = research_features.filter(
    F.col("poll_microseconds") >= split_microseconds
)

display(
    training_features.select(
        F.lit("Training").alias("split"),
        "fetched_at", "bike_target", "dock_target",
    )
    .unionByName(
        validation_features.select(
            F.lit("Validation").alias("split"),
            "fetched_at", "bike_target", "dock_target",
        )
    )
    .groupBy("split")
    .agg(
        F.min("fetched_at").alias("first_poll"),
        F.max("fetched_at").alias("last_poll"),
        F.count("bike_target").alias("bike_examples"),
        F.avg("bike_target").alias("bike_onset_share"),
        F.count("dock_target").alias("dock_examples"),
        F.avg("dock_target").alias("dock_onset_share"),
    )
)

## Logistic Regression Model training and testing

We bring a sample of the training and validation rows into pandas, and fit scikit-learn pipelines, as in
the training notebook. All models use the same features, the same samples, and the same penalty, so the
comparison shows only the effect of the model structure:

1. **Baseline:** the numeric features only.
2. **Geographic:** adds an intercept adjustment for each geographic cell. This is the production model.
3. **Geography × time:** also gives each cell its own daily curve, through interactions between the cell and the time harmonics.

In [0]:
# We list the numeric features, as in the training notebook

shared_features = ["log_capacity", "weekend", "hour_sin_1", "hour_cos_1", "hour_sin_2", "hour_cos_2"]
harmonics = ["hour_sin_1", "hour_cos_1", "hour_sin_2", "hour_cos_2"]

numeric_features = {
    "bike": ["num_bikes_available", "bike_available_fraction",
             "bike_change_30min_per_minute_fraction", *shared_features],
    "dock": ["num_docks_available", "dock_available_fraction",
             "dock_change_30min_per_minute_fraction", *shared_features],
}
all_numeric_features = [*numeric_features["bike"][:3], *numeric_features["dock"][:3], *shared_features]

# Station and hour are not model inputs; the diagnostics use them
model_columns = ["station_id", "lat", "lon", "hour", "geographic_cell",
                 *all_numeric_features, "bike_target", "dock_target"]

training_sample = (training_features
                   .filter(F.col("bike_target").isNotNull() | F.col("dock_target").isNotNull())
                   .select(*model_columns)
                   .sample(fraction=SAMPLE_FRACTION, seed=1).toPandas())
validation_sample = (validation_features
                     .filter(F.col("bike_target").isNotNull() | F.col("dock_target").isNotNull())
                     .select(*model_columns)
                     .sample(fraction=VALIDATION_SAMPLE_FRACTION, seed=1).toPandas())

# Keeping the datatypes safe for pandas/scikit-learn
for sample in [training_sample, validation_sample]:
    sample[all_numeric_features] = sample[all_numeric_features].astype("float64")

training_data = {outcome: training_sample[training_sample[f"{outcome}_target"].notna()].reset_index(drop=True)
                 for outcome in ["bike", "dock"]}
validation_data = {outcome: validation_sample[validation_sample[f"{outcome}_target"].notna()].reset_index(drop=True)
                   for outcome in ["bike", "dock"]}

# Verify that both datasets contain positives and negatives.
for outcome in ["bike", "dock"]:
    for name, frame in [("Training", training_data[outcome]), ("Validation", validation_data[outcome])]:
        if frame[f"{outcome}_target"].nunique() != 2:
            raise ValueError(f"{outcome}: {name} needs both label classes.")
    print(f"{outcome}: {len(training_data[outcome]):,} training rows, "
          f"{len(validation_data[outcome]):,} validation rows")

In [0]:
# Three models get the same evaluation and the same two figures, so the evaluation and the
# figures are functions. We will reuse this code in some way to build a monitoring shiny app


# validation_diagnostics:
# Applies a fitted model to the validation rows, and returns:
#   scores: example count, event rate, mean prediction, log loss, ROC AUC, and the log loss of a
#           constant prediction at the training event rate (the benchmark the model must beat)
#   calibration: observed onset fraction against mean predicted probability, in 20 bins
#   roc: the ROC curve
#   hourly: mean residual (observed minus predicted) by New York hour
#   station: errors by station. Location is included in the grouping to preserve station relocations.

def validation_diagnostics(model, validation, target, training_event_rate):
    # Clip only for numerical stability when taking logarithms.
    probability = np.clip(model.predict_proba(validation)[:, 1], 1e-12, 1 - 1e-12)

    result = validation[["station_id", "lat", "lon", "hour"]].copy()
    result["label"] = validation[target].to_numpy()
    result["probability_value"] = probability
    result["residual"] = result["label"] - result["probability_value"]
    result["log_loss"] = -(result["label"] * np.log(probability)
                           + (1 - result["label"]) * np.log(1 - probability))

    event_rate = result["label"].mean()
    scores = {
        "validation_examples": len(result),
        "event_rate": event_rate,
        "mean_prediction": result["probability_value"].mean(),
        "log_loss": result["log_loss"].mean(),
        "roc_auc": roc_auc_score(result["label"], result["probability_value"]),
        # Constant-probability benchmark: training event frequency.
        "constant_log_loss": float(-event_rate * np.log(training_event_rate)
                                   - (1 - event_rate) * np.log(1 - training_event_rate)),
    }

    calibration = (
        result
        .assign(probability_bin=np.minimum((result["probability_value"] * 20).astype(int), 19))
        .groupby("probability_bin")
        .agg(polls=("label", "size"), predicted=("probability_value", "mean"), observed=("label", "mean"))
        .reset_index()
    )

    false_positive_rate, true_positive_rate, _ = roc_curve(result["label"], result["probability_value"])
    roc = pd.DataFrame({"false_positive_rate": false_positive_rate,
                        "true_positive_rate": true_positive_rate})

    hourly = (
        result.groupby("hour")
        .agg(polls=("label", "size"), observed=("label", "mean"),
             predicted=("probability_value", "mean"), mean_residual=("residual", "mean"))
        .reset_index()
    )

    station = (
        result.groupby(["station_id", "lat", "lon"])
        .agg(polls=("label", "size"), observed=("label", "mean"),
             predicted=("probability_value", "mean"), mean_residual=("residual", "mean"),
             mean_log_loss=("log_loss", "mean"), total_log_loss=("log_loss", "sum"))
        .reset_index()
    )
    station["loss_share_percent"] = 100 * station.total_log_loss / station.total_log_loss.sum()

    return {"scores": scores, "calibration": calibration, "roc": roc,
            "hourly": hourly, "station": station}


# plot_diagnostics:
# One figure for a model: calibration, ROC, and residual by hour, for bikes and docks.

def plot_diagnostics(diagnostics, title, annotation):
    fig, axes = plt.subplots(
        2, 3, figsize=(17, 10), constrained_layout=True
    )

    for row, outcome in enumerate(["bike", "dock"]):
        calibration = diagnostics[outcome]["calibration"]
        roc = diagnostics[outcome]["roc"]
        hourly = diagnostics[outcome]["hourly"]

        ax = axes[row, 0]
        ax.plot([0, 1], [0, 1], "--", color="gray")
        ax.plot(calibration.predicted, calibration.observed, "o-")
        ax.set(
            title=f"{outcome.title()}: calibration",
            xlabel="Mean predicted probability",
            ylabel="Observed onset fraction",
            xlim=(0, 1), ylim=(0, 1),
        )

        ax = axes[row, 1]
        ax.plot(roc.false_positive_rate, roc.true_positive_rate)
        ax.plot([0, 1], [0, 1], "--", color="gray")
        ax.set(
            title=f"{outcome.title()}: ROC",
            xlabel="False-positive rate",
            ylabel="True-positive rate",
            xlim=(0, 1), ylim=(0, 1),
        )

        ax = axes[row, 2]
        ax.plot(hourly.hour, 100 * hourly.mean_residual, "o-")
        ax.axhline(0, color="gray", linestyle="--")
        ax.set(
            title=f"{outcome.title()}: time residual",
            xlabel="New York hour",
            ylabel="Observed minus predicted (percentage points)",
        )

    for ax in axes.flat:
        ax.title.set_fontsize(16)
        ax.xaxis.label.set_size(13)
        ax.yaxis.label.set_size(13)
        ax.tick_params(labelsize=11)

    fig.suptitle(title, fontsize=24, fontweight="bold", x=0.05, y=0.99, ha="left")
    fig.text(
        0.05, 0.94,
        textwrap.fill(" ".join(annotation.split()), width=140),
        fontsize=14, ha="left", va="top", linespacing=1.4,
    )
    fig.get_layout_engine().set(rect=(0, 0, 1, 0.84))
    plt.show()


# plot_error_maps:
# One figure for a model: mean log loss and signed error by station, for bikes and docks.
# loss_reference sets the color scale of the log loss maps, so maps of different models
# use the same scale. The scale is capped at the 95th percentile, not the data itself.

def plot_error_maps(diagnostics, title, annotation, loss_reference):
    fig, axes = plt.subplots(
        2, 2, figsize=(13, 14), constrained_layout=True
    )

    residual_scale = TwoSlopeNorm(vmin=-10, vcenter=0, vmax=10)

    for row, outcome in enumerate(["bike", "dock"]):
        station = diagnostics[outcome]["station"]
        q = station.dropna(subset=["lat", "lon"])

        loss_limit = max(float(loss_reference[outcome]["station"].mean_log_loss.quantile(.95)), 1e-6)
        loss_scale = Normalize(0, loss_limit)

        for ax in axes[row]:
            map_background(ax)

        axes[row, 0].scatter(
            q.lon, q.lat,
            c=q.mean_log_loss,
            cmap="viridis", norm=loss_scale,
            s=10, zorder=2,
        )
        axes[row, 0].set_title(f"{outcome.title()}: mean log loss", fontsize=18)

        colorbar = fig.colorbar(
            ScalarMappable(norm=loss_scale, cmap="viridis"),
            ax=axes[row, 0], shrink=.6, extend="max",
        )
        colorbar.set_label("Mean log loss", fontsize=14)
        colorbar.ax.tick_params(labelsize=12)

        axes[row, 1].scatter(
            q.lon, q.lat,
            c=100 * q.mean_residual,
            cmap="RdBu_r", norm=residual_scale,
            s=10, zorder=2,
        )
        axes[row, 1].set_title(f"{outcome.title()}: signed error", fontsize=18)

    colorbar = fig.colorbar(
        ScalarMappable(norm=residual_scale, cmap="RdBu_r"),
        ax=axes[:, 1].tolist(), shrink=.6, extend="both",
    )
    colorbar.set_label("Observed minus predicted (percentage points)", fontsize=14)
    colorbar.ax.tick_params(labelsize=12)

    fig.suptitle(title, fontsize=24, fontweight="bold", x=0.05, y=0.99, ha="left")
    fig.text(
        0.05, 0.94,
        textwrap.fill(" ".join(annotation.split()), width=140),
        fontsize=14, ha="left", va="top", linespacing=1.4,
    )
    fig.get_layout_engine().set(rect=(0, 0, 1, 0.88))
    plt.show()

## Model One: Baseline logistic regression

The numeric features only. We will use the geographical features later

In [0]:
baseline_models = {}
baseline_diagnostics = {}

for outcome in ["bike", "dock"]:
    model = make_pipeline(
        ColumnTransformer([
            ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), numeric_features[outcome]), # Median impute
        ]),
        LogisticRegression(C=C_PENALTY, max_iter=MAX_ITER), # Logistic regression
    )
    target = training_data[outcome][f"{outcome}_target"]
    baseline_models[outcome] = model.fit(training_data[outcome], target)
    baseline_diagnostics[outcome] = validation_diagnostics(
        model, validation_data[outcome], f"{outcome}_target", target.mean())
    print(outcome, "iterations:", model[-1].n_iter_[0])

display(pd.DataFrame([{"outcome": outcome, **baseline_diagnostics[outcome]["scores"]}
                      for outcome in ["bike", "dock"]]))

## Logistic Model Error Exploration

In [0]:
plot_diagnostics(
    baseline_diagnostics,
    "Baseline logistic regression development validation",
    """Notable residual pattern in prodbability remains, peaking just prior to the morning commute. Model calibration curves suggest 
       very few data points at the upper end of predicted probability.""",
)

In [0]:
plot_error_maps(
    baseline_diagnostics,
    "Geographic errors in the baseline model",
    """The primary concentration to prediction errors for bikes are Harlem/Upper Manhattan, Midtown, 
       and Prospect Park. The major contribution to prediction error for full docks is lower Manhattan and the Upper East Side.
       In both cases the residual plot indicates that the worst predicted regions under-predict the probability of events.""",
    loss_reference=baseline_diagnostics,
)

for outcome in ["bike", "dock"]:
    print(f"{outcome.title()}: largest contributions to validation log loss")
    display(
        baseline_diagnostics[outcome]["station"]
        .sort_values("total_log_loss", ascending=False)
        .head(10)
    )

# Model Variant Two: Geography

We add an intercept adjustment for each geographic cell (one-hot encoded). This is the model that the
training notebook uses in production. Handle unknown lets the model run on cells that are not in the training data.

In [0]:
geographic_models = {}
geographic_diagnostics = {}

for outcome in ["bike", "dock"]:
    model = make_pipeline(
        ColumnTransformer([
            ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), numeric_features[outcome]), # Median impute
            ("cat", OneHotEncoder(handle_unknown="ignore"), ["geographic_cell"]), # One-Hot the geographic cells
        ]),
        LogisticRegression(C=C_PENALTY, max_iter=MAX_ITER), # Logistic regression
    )
    target = training_data[outcome][f"{outcome}_target"]
    geographic_models[outcome] = model.fit(training_data[outcome], target)
    geographic_diagnostics[outcome] = validation_diagnostics(
        model, validation_data[outcome], f"{outcome}_target", target.mean())
    print(outcome, "iterations:", model[-1].n_iter_[0])

display(pd.DataFrame([{"outcome": outcome, **geographic_diagnostics[outcome]["scores"]}
                      for outcome in ["bike", "dock"]]))

In [0]:
plot_diagnostics(
    geographic_diagnostics,
    "Geographic logistic regression: development validation",
    """The production model: one intercept adjustment for each geographic cell.""",
)

In [0]:
plot_error_maps(
    geographic_diagnostics,
    "Geographic errors after adding geographic cells",
    """Same color scale as the baseline maps.""",
    loss_reference=baseline_diagnostics,
)

# Model Variant Three: More geography and time

We build another logistic regression model this time incorporating interactions between geographic boxes and our time features

In [0]:
# The interactions between the cells and the harmonics give each cell its own daily curve.
# PolynomialFeatures(interaction_only=True) multiplies every pair of its input columns. Its input is
# the one-hot cells and the four harmonics, so it makes cell × harmonic columns (the daily curves),
# but also cell × cell columns, which are always zero (a row is in only one cell), and harmonic ×
# harmonic columns. The zero columns have no effect on the fit; they only make the matrix wider.

geo_time_models = {}
geo_time_diagnostics = {}

for outcome in ["bike", "dock"]:
    model = make_pipeline(
        ColumnTransformer([
            ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), numeric_features[outcome]), # Median impute
            ("cell_curves", make_pipeline(
                ColumnTransformer([
                    ("cat", OneHotEncoder(handle_unknown="ignore"), ["geographic_cell"]),
                    ("time", "passthrough", harmonics),
                ]),
                PolynomialFeatures(degree=2, interaction_only=True, include_bias=False),
            ), ["geographic_cell", *harmonics]),
        ]),
        LogisticRegression(C=C_PENALTY, max_iter=MAX_ITER), # Logistic regression
    )
    target = training_data[outcome][f"{outcome}_target"]
    geo_time_models[outcome] = model.fit(training_data[outcome], target)
    geo_time_diagnostics[outcome] = validation_diagnostics(
        model, validation_data[outcome], f"{outcome}_target", target.mean())
    print(outcome, "iterations:", model[-1].n_iter_[0])

display(pd.DataFrame([{"outcome": outcome, **geo_time_diagnostics[outcome]["scores"]}
                      for outcome in ["bike", "dock"]]))

The performance of the geographic-time model improves noticeably above the baseline logistic regression model.

In [0]:
plot_diagnostics(
    geo_time_diagnostics,
    "Geography × time logistic regression: development validation",
    """This model improves upon the overal performance of the baseline model, but the residual patterns are quite similar. Calibration
       is excellent and even maintains at the high end where there are relatively few examples.""",
)

In [0]:
plot_error_maps(
    geo_time_diagnostics,
    "Geographic errors after adding local daily patterns",
    """Despite the addition of geographic features, the same residual pattern persists""",
    loss_reference=baseline_diagnostics,
)

In [0]:
# Compare validation performance.

all_diagnostics = {
    "Baseline LR": baseline_diagnostics,
    "Geographic LR": geographic_diagnostics,
    "Geography × time LR": geo_time_diagnostics,
}

comparison = pd.DataFrame([
    {"outcome": outcome, "model": model_name, **diagnostics[outcome]["scores"]}
    for model_name, diagnostics in all_diagnostics.items()
    for outcome in ["bike", "dock"]
])

display(
    comparison[
        [
            "outcome", "model", "validation_examples",
            "event_rate", "log_loss", "constant_log_loss", "roc_auc",
        ]
    ].sort_values(["outcome", "log_loss"])
)

# Overlay the ROC curves.
fig, axes = plt.subplots(
    1, 2, figsize=(12, 5), constrained_layout=True
)

for ax, outcome in zip(axes, ["bike", "dock"]):
    for model_name, diagnostics in all_diagnostics.items():
        roc = diagnostics[outcome]["roc"]
        auc = diagnostics[outcome]["scores"]["roc_auc"]
        ax.plot(
            roc.false_positive_rate, roc.true_positive_rate,
            label=f"{model_name}: AUC {auc:.4f}",
            linewidth=2,
        )

    ax.plot([0, 1], [0, 1], "--", color="gray")
    ax.set(
        title=f"{outcome.title()} stockout onset",
        xlabel="False-positive rate",
        ylabel="True-positive rate",
        xlim=(0, 1),
        ylim=(0, 1),
    )
    ax.legend(fontsize=10)
    ax.title.set_fontsize(16)

fig.suptitle(
    "Model comparison: development validation",
    fontsize=22,
    fontweight="bold",
)
plt.show()


## Conclusions

1. There is persistent temporal and geographic heterogeneity in both stockout and full dock risk and in station capacity.
2. There are some regions where stockout risk occurs at all times of the day, though an obvious commutter pattern can be seen throughout Manhattan.
3. Our Georgraphic/Time interaction Logistic Regression model which incorporate (naively constructed) geographic features show improved performance but had identical patterns of residuals.

The goal of this development exercise was to create to a first exemplar to show as a production model, so we will select the geographic logistic regression. In production it uses one intercept adjustment for each geographic cell, without the cell-specific daily curves, and its penalty is chosen by cross-validation in the training notebook. I think there is significant room for model improvement and the dataset is quite short. A goal is to develop a hierarchical model which can even better leverage the geographic structure.